In [19]:
from pathlib import Path

import joblib


In [21]:

# Chemin absolu ou relatif vers ton fichier .joblib
JOBLIB_PATH = Path(r"C:\Users\bapti\Desktop\Archive_Baptiste\LTE\masclab_codeReview\pyVSmat\GT\raw_images\Valais-2016_PROCESSED\2016.12.19\02\DATA\GOOD\2016.12.19_02.00.40_flake_16_cam0.joblib")

obj = joblib.load(JOBLIB_PATH)
print("path:", JOBLIB_PATH.resolve())
print("type:", type(obj))

if isinstance(obj, dict):
    print("keys (top-level):", sorted(obj.keys())[:30], "..." if len(obj) > 30 else "")
else:
    print("repr:", repr(obj)[:500])

path: C:\Users\bapti\Desktop\Archive_Baptiste\LTE\masclab_codeReview\pyVSmat\GT\raw_images\Valais-2016_PROCESSED\2016.12.19\02\DATA\GOOD\2016.12.19_02.00.40_flake_16_cam0.joblib
type: <class 'dict'>
keys (top-level): ['C_out', 'D90', 'Dmax', 'DmaxA', 'DmaxB', 'Dmax_theta', 'Dmean', 'E', 'E_in', 'E_out', 'F', 'F_jac', 'H', 'Rect', 'Sym', 'area', 'area2', 'area_focus', 'area_focus_ratio', 'area_lap', 'area_porous', 'area_range', 'bw_mask', 'bw_mask_filled', 'bw_perim', 'cam', 'centroid', 'centroid_global', 'centroid_local', 'centroid_local_init'] ...


In [17]:
# Nombre total de champs = feuilles du dict imbriqué (équivalent « nombre de colonnes » d’un DF aplati)
from typing import Any, Mapping


def iter_leaf_paths(d: Mapping[str, Any], prefix: str = "") -> list[str]:
    paths: list[str] = []
    for k, v in d.items():
        p = f"{prefix}.{k}" if prefix else str(k)
        if isinstance(v, Mapping) and v:
            paths.extend(iter_leaf_paths(v, p))
        else:
            paths.append(p)
    return paths


if isinstance(obj, dict):
    leaves = iter_leaf_paths(obj)
    print("Clés de 1er niveau:", len(obj))
    print("Nombre total de clés (feuilles, chemins pointés):", len(leaves))
    print("Exemples:", leaves[:12], "..." if len(leaves) > 12 else "")
else:
    print("Pas un dict : une seule « variable » au sens aplati.")

Clés de 1er niveau: 75
Nombre total de clés (feuilles, chemins pointés): 118
Exemples: ['data', 'bw_mask', 'bw_mask_filled', 'nb_holes', 'holes_mask', 'y', 'x', 'area', 'area2', 'area_porous', 'mean_intens', 'max_intens'] ...


In [22]:
# Visualisation type DataFrame : une ligne par feuille (chemin, type, aperçu)
# (autonome : exécute au minimum la cellule de chargement `obj` avant.)
from typing import Any, Mapping

import numpy as np
import pandas as pd


def _iter_leaf_paths(d: Mapping[str, Any], prefix: str = "") -> list[str]:
    paths: list[str] = []
    for k, v in d.items():
        p = f"{prefix}.{k}" if prefix else str(k)
        if isinstance(v, Mapping) and v:
            paths.extend(_iter_leaf_paths(v, p))
        else:
            paths.append(p)
    return paths


def _get_at(d: dict, dotted: str):
    cur = d
    for part in dotted.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return None
        cur = cur[part]
    return cur


def _aperçu(v):
    if isinstance(v, np.ndarray):
        if v.size <= 12 and v.ndim <= 1:
            return str(v.tolist())
        return f"ndarray {v.shape} {v.dtype}"
    if isinstance(v, (np.floating, np.integer)):
        try:
            return v.item()
        except Exception:
            return repr(v)
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    s = repr(v)
    return s if len(s) <= 120 else s[:117] + "..."


if isinstance(obj, dict):
    rows = []
    for path in sorted(_iter_leaf_paths(obj)):
        v = _get_at(obj, path)
        rows.append(
            {"chemin": path, "type_python": type(v).__name__, "aperçu": _aperçu(v)}
        )
    df_roi = pd.DataFrame(rows)
    display(df_roi)
    print("shape (lignes, colonnes):", df_roi.shape)
else:
    print("Pas un dict, pas de tableau aplati.")

,chemin,type_python,aperçu
0,C_out.A,float,748.693966
1,C_out.X0,float64,14.750745
2,C_out.Y0,float64,9.403518
3,C_out.r,float,15.437509
4,D90,float,21.080113
...,...,...,...
113,x_perim,ndarray,"ndarray (74, 1) int32"
114,xhi,float64,9.248315
115,y,ndarray,"ndarray (472, 1) int64"
116,y_loc,int,496


shape (lignes, colonnes): (118, 3)
